# Experiment: Real GW Event Inference (GW170817 + GW190425)

Objective:
- End-to-end pipeline on real-event transients for GW170817 and GW190425.
- Time window: `t0-30d` to `t0+60d`.
- Region filter: keep candidates with `credible_level <= 0.9`.
- Convert light curves to sparse-padded model input (`seq_len=90`) and run `supcon_v2` inference.

Notes:
- This notebook is cache-first. Online steps always try local cache before network.
- If network is unavailable, online steps print `SKIPPED: network unavailable` and continue.
- The notebook uses absolute event times:
  - `GW170817`: `2017-08-17T12:41:04.444` (`MJD 57982.52852366262`)
  - `GW190425`: `2019-04-25T08:18:42.019` (`MJD 58598.34631965919`)


## 0. Goals And Configuration


In [ ]:
from __future__ import annotations

import math
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from astropy.time import Time

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path('<BASE_DIR>').resolve()
MODEL_DIR = PROJECT_ROOT / 'ML+GW+KN' / 'Model'
NOTEBOOK_DIR = PROJECT_ROOT / 'ML+GW+KN' / 'Model' / 'notebook'
CACHE_ROOT = PROJECT_ROOT / 'data' / 'real_events'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

WINDOW_PRE_DAYS = 10.0
WINDOW_POST_DAYS = 20.0
CREDIBLE_LEVEL_MAX = 0.90
SEQ_LEN = 90
FORCE_REFRESH = True
ENABLE_OFFLINE_MOCK_IF_EMPTY = False
RUN_OFFLINE_STRUCT_TESTS = False
REAL_DATA_ONLY = True

# Candidate retrieval speed controls (quick mode by default).
FAST_CANDIDATE_QUERY = False
ALERCE_MAX_PAGES = 20 if FAST_CANDIDATE_QUERY else 100
ALERCE_PAGE_SIZE = 500 if FAST_CANDIDATE_QUERY else 1000
ALERCE_TIMEOUT_S = 12.0 if FAST_CANDIDATE_QUERY else 30.0
CREDIBLE_CHUNK_SIZE = 512 if FAST_CANDIDATE_QUERY else 1024
MAX_CANDIDATES_PER_EVENT = 3000 if FAST_CANDIDATE_QUERY else None
MIN_DETECTIONS = 2
STRICT_WINDOW_TIME_FILTER = True
STRICT_WINDOW_MIN_DETECTIONS = True
ZTF_LC_BACKEND = 'alerce'
ZTF_LC_TIMEOUT_S = 20.0
ENABLE_CANDIDATE_PROGRESS = True

LSST_BANDS = ['LSST-u', 'LSST-g', 'LSST-r', 'LSST-i', 'LSST-z', 'LSST-Y']
BAND_TO_INDEX = {band: idx for idx, band in enumerate(LSST_BANDS)}


@dataclass
class EventConfig:
    event_name: str
    t0_mjd: float
    t0_iso: str
    gw_scalar_source: str
    skymap_path: str | None
    catalog_backend: str
    posterior_path: str | None = None


GW190425_LOCAL_DIR = PROJECT_ROOT / 'data' / 'GW190425'

EVENT_CONFIGS = {
    'GW170817': EventConfig(
        event_name='GW170817',
        t0_mjd=57982.52852366262,
        t0_iso='2017-08-17T12:41:04.444',
        gw_scalar_source='data/GW170817A/GW170817_GWTC-1.hdf5 (local)',
        skymap_path=str(PROJECT_ROOT / 'data' / 'GW170817A' / 'bayestar_no_virgo.fits'),
        catalog_backend='open_catalog',
    ),
    'GW190425': EventConfig(
        event_name='GW190425',
        t0_mjd=58598.34631965919,
        t0_iso='2019-04-25T08:18:42.019',
        gw_scalar_source='data/GW190425/posterior_samples.h5 + bayestar.fits (local)',
        skymap_path=str(GW190425_LOCAL_DIR / 'bayestar.fits'),
        posterior_path=str(GW190425_LOCAL_DIR / 'posterior_samples.h5'),
        catalog_backend='alerce',
    ),
}


print(
    f"Candidate query config: fast={FAST_CANDIDATE_QUERY}, "
    f"max_pages={ALERCE_MAX_PAGES}, page_size={ALERCE_PAGE_SIZE}, "
    f"timeout_s={ALERCE_TIMEOUT_S}, max_candidates={MAX_CANDIDATES_PER_EVENT}, "
    f"min_detections={MIN_DETECTIONS}, strict_window={STRICT_WINDOW_TIME_FILTER}, "
    f"ztf_backend={ZTF_LC_BACKEND}, real_data_only={REAL_DATA_ONLY}, "
    f"run_offline_struct_tests={RUN_OFFLINE_STRUCT_TESTS}"
)

for name, cfg in EVENT_CONFIGS.items():
    start = cfg.t0_mjd - WINDOW_PRE_DAYS
    end = cfg.t0_mjd + WINDOW_POST_DAYS
    print(
        f"{name}: t0={cfg.t0_iso} | "
        f"window=[{Time(start, format='mjd').isot}, {Time(end, format='mjd').isot}]"
    )


## 1. Dependencies And Paths


In [ ]:
import importlib.util
import sys
import json
import os
import socket
import warnings
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import h5py
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F
from astropy.io import fits

warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

# Ensure Model package imports work inside Jupyter (for ALBEF_train -> data_loader).
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))


def load_module_from_path(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot import module from {module_path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


DATA_LOADER = load_module_from_path('data_loader_runtime', MODEL_DIR / 'data_loader.py')
MODEL_LIB = load_module_from_path('model_runtime', MODEL_DIR / 'model.py')

# Expose canonical module names expected by ALBEF_train.py imports.
sys.modules.setdefault('data_loader', DATA_LOADER)
sys.modules.setdefault('model', MODEL_LIB)

TRAIN_LIB = load_module_from_path('albef_train_runtime', MODEL_DIR / 'ALBEF_train.py')
ZTF_SCRIPT = load_module_from_path(
    'ztf_lightcurve_runtime',
    PROJECT_ROOT / 'skills' / 'ztf-lightcurve-analysis' / 'scripts' / 'ztf_lightcurve.py',
)

sample_moc_skymap = DATA_LOADER.sample_moc_skymap
compute_credible_level = TRAIN_LIB.compute_credible_level
GWOpticalALBEFModel = MODEL_LIB.GWOpticalALBEFModel


def network_available(hosts: list[str] | None = None) -> bool:
    hosts = hosts or ['api.alerce.online', 'api.astrocats.space', 'gwosc.org']
    for host in hosts:
        try:
            socket.gethostbyname(host)
            return True
        except OSError:
            continue
    return False


def http_json(url: str, params: dict[str, Any] | None = None, timeout_s: float = 30.0) -> Any:
    full_url = url
    if params:
        query = urlencode({k: v for k, v in params.items() if v is not None})
        sep = '&' if '?' in url else '?'
        full_url = f'{url}{sep}{query}'
    request = Request(full_url, headers={'User-Agent': 'codex-gw-real-inference/1.0'})
    try:
        with urlopen(request, timeout=timeout_s) as response:
            return json.loads(response.read().decode('utf-8'))
    except HTTPError as exc:
        raise RuntimeError(f'HTTP {exc.code} for {full_url}') from exc
    except URLError as exc:
        raise RuntimeError(f'Network error for {full_url}: {exc}') from exc


def download_binary(url: str, out_path: Path, timeout_s: float = 60.0) -> Path:
    request = Request(url, headers={'User-Agent': 'codex-gw-real-inference/1.0'})
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(request, timeout=timeout_s) as response:
        out_path.write_bytes(response.read())
    return out_path


print('Loaded modules: data_loader, model, ALBEF_train, ztf_lightcurve script')
print(f'Network available: {network_available()}')


## 2. Event Metadata And Skymap


In [ ]:
from typing import Optional


def recursive_extract_urls(payload: Any) -> list[str]:
    urls: list[str] = []
    if isinstance(payload, dict):
        for _, value in payload.items():
            urls.extend(recursive_extract_urls(value))
    elif isinstance(payload, list):
        for item in payload:
            urls.extend(recursive_extract_urls(item))
    elif isinstance(payload, str):
        low = payload.lower()
        if low.startswith('http') and ('.fits' in low or '.fit' in low):
            urls.append(payload)
    return urls


def select_skymap_url(urls: list[str]) -> str | None:
    if not urls:
        return None
    preferred = sorted(urls, key=lambda u: (('bayestar' not in u.lower()), len(u)))
    return preferred[0]


def load_gw170817_scalar_and_skymap(cfg: EventConfig) -> tuple[torch.Tensor, torch.Tensor, dict[str, Any]]:
    h5_path = PROJECT_ROOT / 'data' / 'GW170817A' / 'GW170817_GWTC-1.hdf5'
    if not h5_path.exists():
        raise FileNotFoundError(h5_path)

    with h5py.File(h5_path, 'r') as f:
        lowspin = f['/IMRPhenomPv2NRT_lowSpin_posterior'][:]

    spin1 = lowspin['spin1']
    spin2 = lowspin['spin2']
    m1 = lowspin['m1_detector_frame_Msun']
    m2 = lowspin['m2_detector_frame_Msun']
    cos_inc = lowspin['costheta_jn']

    _, meta = DATA_LOADER.read_sky_map(cfg.skymap_path, nest=True)
    dist_mean = float(meta.get('distmean', np.nan))
    dist_std = float(meta.get('diststd', np.nan))

    gw_scalar = torch.tensor(
        [
            float(np.mean(m1)),
            float(np.mean(m2)),
            float(np.mean(spin1)),
            float(np.mean(spin2)),
            float(np.mean(cos_inc)),
            dist_mean / 1000.0,
            dist_std / 1000.0,
        ],
        dtype=torch.float32,
    )

    gw_moc = sample_moc_skymap(cfg.skymap_path).to(torch.float32)
    meta_out = {
        'distmean_mpc': dist_mean,
        'diststd_mpc': dist_std,
        'skymap_path': cfg.skymap_path,
        'gw_scalar_source': cfg.gw_scalar_source,
    }
    return gw_scalar, gw_moc, meta_out


def _select_existing_file(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None


def load_or_download_gw190425_skymap(cfg: EventConfig) -> tuple[Path | None, dict[str, Any]]:
    # 1) explicit local path from config
    explicit = Path(cfg.skymap_path) if cfg.skymap_path else None
    if explicit is not None and explicit.exists():
        return explicit, {'source': 'local_explicit_path'}

    # 2) local manual folder
    local_dir = PROJECT_ROOT / 'data' / 'GW190425'
    local_fits = sorted(local_dir.glob('*.fits')) + sorted(local_dir.glob('*.fit'))
    if local_fits:
        return local_fits[0], {'source': 'local_manual_dir', 'event_dir': str(local_dir)}

    # 3) cache folder
    event_dir = CACHE_ROOT / cfg.event_name
    event_dir.mkdir(parents=True, exist_ok=True)
    cached_fits = sorted(event_dir.glob('*.fits')) + sorted(event_dir.glob('*.fit'))
    if cached_fits:
        return cached_fits[0], {'source': 'local_cache', 'event_dir': str(event_dir)}

    # 4) online fallback
    if not network_available(['gwosc.org']):
        print('SKIPPED: network unavailable (GWOSC)')
        return None, {'source': 'network_unavailable'}

    api_candidates = [
        'https://gwosc.org/eventapi/jsonfull/GWTC-2.1-confident/',
        'https://gwosc.org/eventapi/jsonfull/GWTC-3-confident/',
    ]

    payload = None
    for api_url in api_candidates:
        try:
            payload = http_json(api_url)
            break
        except Exception as exc:
            print(f'GWOSC API failed: {api_url} ({exc})')

    if payload is None:
        return None, {'source': 'gwosc_api_failed'}

    urls = [u for u in recursive_extract_urls(payload) if '190425' in u.lower()]
    skymap_url = select_skymap_url(urls)
    if skymap_url is None:
        print('GW190425 skymap URL not found in GWOSC payload.')
        return None, {'source': 'skymap_url_not_found'}

    event_dir = CACHE_ROOT / cfg.event_name
    event_dir.mkdir(parents=True, exist_ok=True)
    local_name = skymap_url.split('/')[-1] or 'GW190425_skymap.fits'
    if not local_name.lower().endswith(('.fits', '.fit')):
        local_name += '.fits'
    local_path = event_dir / local_name

    try:
        download_binary(skymap_url, local_path)
    except Exception as exc:
        print(f'Failed to download GW190425 skymap: {exc}')
        return None, {'source': 'download_failed', 'url': skymap_url}

    return local_path, {'source': 'gwosc_download', 'url': skymap_url}


def load_local_gw190425_posterior(cfg: EventConfig) -> tuple[dict[str, float], dict[str, Any]] | None:
    local_dir = PROJECT_ROOT / 'data' / 'GW190425'
    candidates = []
    if cfg.posterior_path:
        candidates.append(Path(cfg.posterior_path))
    candidates.append(local_dir / 'posterior_samples.h5')

    posterior_path = _select_existing_file(candidates)
    if posterior_path is None:
        return None

    pref_groups = [
        'PhenomPNRT-HS',
        'PhenomDNRT-HS',
        'PhenomPNRT-LS',
        'PhenomDNRT-LS',
        'TaylorF2-HS',
        'TaylorF2-LS',
    ]

    with h5py.File(posterior_path, 'r') as f:
        group_name = None
        samples = None
        for g in pref_groups:
            if g in f and 'posterior_samples' in f[g]:
                group_name = g
                samples = f[g]['posterior_samples'][:]
                break

        if samples is None:
            # fallback: first group containing posterior_samples
            for k in f.keys():
                if isinstance(f[k], h5py.Group) and 'posterior_samples' in f[k]:
                    group_name = k
                    samples = f[k]['posterior_samples'][:]
                    break

        if samples is None:
            raise RuntimeError(f'No posterior_samples found in {posterior_path}')

    names = set(samples.dtype.names or [])

    def _col(preferred: list[str], default: float = np.nan) -> np.ndarray:
        for n in preferred:
            if n in names:
                arr = np.asarray(samples[n], dtype=np.float64)
                arr = arr[np.isfinite(arr)]
                if arr.size > 0:
                    return arr
        return np.array([default], dtype=np.float64)

    m1 = _col(['mass_1', 'mass_1_source'])
    m2 = _col(['mass_2', 'mass_2_source'])
    s1 = _col(['spin_1z', 'a_1'], default=0.02)
    s2 = _col(['spin_2z', 'a_2'], default=0.02)
    cos_inc = _col(['cos_theta_jn', 'cos_iota'], default=math.cos(1.1))
    d_l = _col(['luminosity_distance'], default=156.0)

    stats = {
        'mass1': float(np.nanmean(m1)),
        'mass2': float(np.nanmean(m2)),
        'spin1z': float(np.nanmean(s1)),
        'spin2z': float(np.nanmean(s2)),
        'cos_inclination': float(np.nanmean(cos_inc)),
        'luminosity_distance_mean_mpc': float(np.nanmean(d_l)),
        'luminosity_distance_std_mpc': float(np.nanstd(d_l)),
    }

    meta = {
        'posterior_path': str(posterior_path),
        'posterior_group': group_name,
        'n_posterior_samples': int(len(samples)),
    }
    return stats, meta


def load_gw190425_scalar_and_skymap(cfg: EventConfig) -> tuple[torch.Tensor, torch.Tensor, dict[str, Any]] | None:
    skymap_path, skymap_meta_info = load_or_download_gw190425_skymap(cfg)
    if skymap_path is None or not skymap_path.exists():
        return None

    _, skymap_meta = DATA_LOADER.read_sky_map(str(skymap_path), nest=True)
    dist_mean_map = float(skymap_meta.get('distmean', np.nan))
    dist_std_map = float(skymap_meta.get('diststd', np.nan))

    posterior_loaded = load_local_gw190425_posterior(cfg)
    if posterior_loaded is not None:
        posterior_stats, posterior_meta = posterior_loaded
        m1 = posterior_stats['mass1']
        m2 = posterior_stats['mass2']
        s1 = posterior_stats['spin1z']
        s2 = posterior_stats['spin2z']
        cos_inc = posterior_stats['cos_inclination']
        dist_mean_post = posterior_stats['luminosity_distance_mean_mpc']
        dist_std_post = posterior_stats['luminosity_distance_std_mpc']
    else:
        posterior_meta = {'posterior_path': None, 'posterior_group': None, 'n_posterior_samples': 0}
        m1, m2, s1, s2, cos_inc = 1.60, 1.45, 0.02, 0.02, math.cos(1.1)
        dist_mean_post, dist_std_post = 156.0, 40.0

    dist_mean = dist_mean_map if np.isfinite(dist_mean_map) else dist_mean_post
    dist_std = dist_std_map if np.isfinite(dist_std_map) else dist_std_post

    gw_scalar = torch.tensor(
        [m1, m2, s1, s2, cos_inc, dist_mean / 1000.0, dist_std / 1000.0],
        dtype=torch.float32,
    )

    gw_moc = sample_moc_skymap(str(skymap_path)).to(torch.float32)
    meta_out = {
        **skymap_meta_info,
        **posterior_meta,
        'distmean_mpc': dist_mean,
        'diststd_mpc': dist_std,
        'skymap_path': str(skymap_path),
        'gw_scalar_source': cfg.gw_scalar_source,
    }
    return gw_scalar, gw_moc, meta_out


def build_event_payload(cfg: EventConfig) -> dict[str, Any] | None:
    if cfg.event_name == 'GW170817':
        gw_scalar, gw_moc, meta_info = load_gw170817_scalar_and_skymap(cfg)
    elif cfg.event_name == 'GW190425':
        loaded = load_gw190425_scalar_and_skymap(cfg)
        if loaded is None:
            return None
        gw_scalar, gw_moc, meta_info = loaded
    else:
        raise ValueError(f'Unsupported event: {cfg.event_name}')

    assert gw_scalar.shape == (7,), f'{cfg.event_name}: unexpected gw_scalar shape {gw_scalar.shape}'
    assert gw_moc.shape[0] == 7, f'{cfg.event_name}: unexpected gw_moc channels {gw_moc.shape}'

    start_mjd = cfg.t0_mjd - WINDOW_PRE_DAYS
    end_mjd = cfg.t0_mjd + WINDOW_POST_DAYS

    return {
        'config': cfg,
        'gw_scalar': gw_scalar,
        'gw_moc': gw_moc,
        'window_start_mjd': start_mjd,
        'window_end_mjd': end_mjd,
        'meta': meta_info,
    }


event_payloads: dict[str, dict[str, Any]] = {}
for event_name, cfg in EVENT_CONFIGS.items():
    payload = build_event_payload(cfg)
    if payload is None:
        print(f'{event_name}: unavailable (missing skymap or posterior)')
        continue
    event_payloads[event_name] = payload
    print(
        f"{event_name}: gw_scalar={tuple(payload['gw_scalar'].shape)}, "
        f"gw_moc={tuple(payload['gw_moc'].shape)}, "
        f"skymap={payload['meta'].get('skymap_path')}"
    )
    if event_name == 'GW190425':
        print(
            f"GW190425 posterior={payload['meta'].get('posterior_path')}, "
            f"group={payload['meta'].get('posterior_group')}, "
            f"n={payload['meta'].get('n_posterior_samples')}"
        )


## 3. Online Candidate Retrieval


In [ ]:
from typing import Iterable
import time


CANDIDATE_COLUMNS = [
    'event_name',
    'object_id',
    'survey',
    'ra',
    'dec',
    'first_mjd',
    'last_mjd',
    'time_window_overlap',
    'n_detections',
    'credible_level',
    'in_90_region',
]


def _extract_list_payload(payload: Any) -> list[dict[str, Any]]:
    if payload is None:
        return []
    if isinstance(payload, list):
        return [item for item in payload if isinstance(item, dict)]
    if isinstance(payload, dict):
        for key in ('items', 'results', 'objects', 'data', 'candidates', 'result'):
            if key in payload:
                return _extract_list_payload(payload[key])
        if payload and all(isinstance(v, (list, tuple)) for v in payload.values()):
            max_len = max(len(v) for v in payload.values())
            rows = []
            for idx in range(max_len):
                row = {k: (v[idx] if idx < len(v) else None) for k, v in payload.items()}
                rows.append(row)
            return rows
    return []


def _first_float(row: dict[str, Any], keys: Iterable[str]) -> float | None:
    for key in keys:
        if key not in row:
            continue
        value = row.get(key)
        if value in (None, '', 'nan', 'NaN', 'null', 'None'):
            continue
        try:
            return float(value)
        except Exception:
            continue
    return None


def _first_int(row: dict[str, Any], keys: Iterable[str]) -> int | None:
    for key in keys:
        if key not in row:
            continue
        value = row.get(key)
        if value in (None, '', 'nan', 'NaN', 'null', 'None'):
            continue
        try:
            return int(float(value))
        except Exception:
            continue
    return None


def compute_credible_levels(
    gw_moc: torch.Tensor,
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    chunk: int = CREDIBLE_CHUNK_SIZE,
    show_progress: bool = ENABLE_CANDIDATE_PROGRESS,
) -> np.ndarray:
    if len(ra_deg) == 0:
        return np.array([], dtype=np.float32)

    n_total = len(ra_deg)
    cred = np.zeros(n_total, dtype=np.float32)
    t0 = time.perf_counter()
    n_chunks = (n_total + chunk - 1) // chunk

    for chunk_idx, start in enumerate(range(0, n_total, chunk), start=1):
        end = min(start + chunk, n_total)
        coords = torch.tensor(np.stack([ra_deg[start:end], dec_deg[start:end]], axis=1), dtype=torch.float32)
        gw_batch = gw_moc.unsqueeze(0).repeat(end - start, 1, 1)
        with torch.no_grad():
            cred_chunk = compute_credible_level(gw_batch, coords).squeeze(-1).cpu().numpy()
        cred[start:end] = cred_chunk

        if show_progress and (chunk_idx == 1 or chunk_idx % 10 == 0 or chunk_idx == n_chunks):
            elapsed = time.perf_counter() - t0
            done = end
            rate = done / elapsed if elapsed > 0 else 0.0
            eta = (n_total - done) / rate if rate > 0 else float('inf')
            print(
                f"credible-level: {done}/{n_total} ({100.0 * done / n_total:.1f}%), "
                f"elapsed={elapsed:.1f}s, eta={eta:.1f}s"
            )

    if show_progress:
        elapsed = time.perf_counter() - t0
        print(f"credible-level done: n={n_total}, elapsed={elapsed:.1f}s")
    return cred



def query_alerce_candidates(
    start_mjd: float,
    end_mjd: float,
    max_pages: int = ALERCE_MAX_PAGES,
    page_size: int = ALERCE_PAGE_SIZE,
    timeout_s: float = ALERCE_TIMEOUT_S,
    show_progress: bool = ENABLE_CANDIDATE_PROGRESS,
) -> pd.DataFrame:
    base_urls = [
        'https://api.alerce.online/ztf/v1/objects',
        'https://api.alerce.online/ztf/v1/objects/',
        'https://api.alerce.online/ztf/v1/objects/query',
    ]

    rows: list[dict[str, Any]] = []
    for base_url in base_urls:
        backend_rows: list[dict[str, Any]] = []
        failed = False
        t_backend = time.perf_counter()

        if show_progress:
            print(
                f"ALeRCE backend: {base_url} | pages<= {max_pages}, "
                f"page_size={page_size}, timeout={timeout_s}s"
            )

        for page in range(1, max_pages + 1):
            params = {
                'firstmjd': start_mjd,
                'lastmjd': end_mjd,
                'page': page,
                'page_size': page_size,
            }
            t_page = time.perf_counter()
            try:
                payload = http_json(base_url, params=params, timeout_s=timeout_s)
            except Exception as exc:
                failed = True
                if show_progress:
                    print(f"  page {page}: failed ({exc})")
                break

            chunk_rows = _extract_list_payload(payload)
            page_elapsed = time.perf_counter() - t_page
            n_chunk = len(chunk_rows)
            if show_progress:
                print(
                    f"  page {page}: rows={n_chunk}, total={len(backend_rows) + n_chunk}, "
                    f"elapsed={page_elapsed:.2f}s"
                )

            if not chunk_rows:
                break

            backend_rows.extend(chunk_rows)
            if n_chunk < page_size:
                break

        if show_progress:
            print(
                f"ALeRCE backend finished: rows={len(backend_rows)}, "
                f"elapsed={time.perf_counter() - t_backend:.1f}s"
            )

        if backend_rows:
            rows = backend_rows
            break
        if not failed:
            break

    if not rows:
        return pd.DataFrame(columns=['object_id', 'survey', 'ra', 'dec', 'first_mjd', 'last_mjd', 'n_detections'])

    normalized: list[dict[str, Any]] = []
    for row in rows:
        low = {str(k).lower(): v for k, v in row.items()}
        oid = low.get('oid') or low.get('objectid') or low.get('name') or low.get('id')
        if oid is None:
            continue
        ra = _first_float(low, ['meanra', 'ra', 'objra'])
        dec = _first_float(low, ['meandec', 'dec', 'objdec'])
        if ra is None or dec is None:
            continue
        first_mjd = _first_float(low, ['firstmjd', 'mjdmin', 'first_detection'])
        last_mjd = _first_float(low, ['lastmjd', 'mjdmax', 'last_detection'])
        n_detections = _first_int(low, ['ndet', 'ndethist', 'nobs', 'n_det', 'detections'])
        normalized.append(
            {
                'object_id': str(oid),
                'survey': 'ZTF',
                'ra': ra,
                'dec': dec,
                'first_mjd': first_mjd,
                'last_mjd': last_mjd,
                'n_detections': n_detections,
            }
        )

    if not normalized:
        return pd.DataFrame(columns=['object_id', 'survey', 'ra', 'dec', 'first_mjd', 'last_mjd', 'n_detections'])

    df = pd.DataFrame(normalized).drop_duplicates(subset=['object_id']).reset_index(drop=True)
    if show_progress:
        print(f"ALeRCE normalized unique candidates: {len(df)}")
    return df



def query_open_catalog_candidates(start_mjd: float, end_mjd: float, force_include_at2017gfo: bool = True) -> pd.DataFrame:
    # Open catalog API shapes vary by deployment; try several query patterns.
    endpoint_patterns = [
        (
            'https://api.astrocats.space/catalog',
            {
                'format': 'json',
                'time_min': Time(start_mjd, format='mjd').isot,
                'time_max': Time(end_mjd, format='mjd').isot,
            },
        ),
        (
            'https://api.astrocats.space/catalog',
            {
                'format': 'json',
                'min_date': Time(start_mjd, format='mjd').isot,
                'max_date': Time(end_mjd, format='mjd').isot,
            },
        ),
    ]

    rows: list[dict[str, Any]] = []
    for url, params in endpoint_patterns:
        try:
            payload = http_json(url, params=params)
        except Exception:
            continue

        list_rows = _extract_list_payload(payload)
        for row in list_rows:
            low = {str(k).lower(): v for k, v in row.items()}
            obj_id = low.get('name') or low.get('objectid') or low.get('id')
            ra = _first_float(low, ['ra', 'meanra', 'objra'])
            dec = _first_float(low, ['dec', 'meandec', 'objdec'])
            if obj_id is None or ra is None or dec is None:
                continue
            first_mjd = _first_float(low, ['mjdmin', 'firstmjd'])
            last_mjd = _first_float(low, ['mjdmax', 'lastmjd'])
            n_detections = _first_int(low, ['ndet', 'nobs', 'n_det', 'detections', 'nphot'])
            rows.append(
                {
                    'object_id': str(obj_id),
                    'survey': str(low.get('survey', low.get('catalog', 'OpenCatalog'))),
                    'ra': ra,
                    'dec': dec,
                    'first_mjd': first_mjd,
                    'last_mjd': last_mjd,
                    'n_detections': n_detections,
                }
            )

        if rows:
            break

    df = pd.DataFrame(rows, columns=['object_id', 'survey', 'ra', 'dec', 'first_mjd', 'last_mjd', 'n_detections'])

    if force_include_at2017gfo:
        at2017gfo = pd.DataFrame(
            [
                {
                    'object_id': 'AT2017gfo',
                    'survey': 'OpenCatalog',
                    'ra': 197.450374,
                    'dec': -23.381495,
                    'first_mjd': 57982.53,
                    'last_mjd': 58005.0,
                    'n_detections': 2,
                }
            ]
        )
        df = pd.concat([df, at2017gfo], ignore_index=True)

    if len(df) == 0:
        return pd.DataFrame(columns=['object_id', 'survey', 'ra', 'dec', 'first_mjd', 'last_mjd', 'n_detections'])

    df = df.drop_duplicates(subset=['object_id']).reset_index(drop=True)
    return df


def build_candidates_for_event(event_name: str, force_refresh: bool = FORCE_REFRESH) -> pd.DataFrame:
    payload = event_payloads[event_name]
    cfg: EventConfig = payload['config']

    event_dir = CACHE_ROOT / event_name
    event_dir.mkdir(parents=True, exist_ok=True)
    candidate_csv = event_dir / f'candidates_{event_name}.csv'

    if candidate_csv.exists() and not force_refresh:
        cached = pd.read_csv(candidate_csv)
        expected = set(CANDIDATE_COLUMNS)
        missing = [col for col in expected if col not in cached.columns]
        if not missing:
            print(f'{event_name}: loaded candidates from cache ({candidate_csv})')
            return cached

    if not network_available():
        print(f'{event_name}: SKIPPED: network unavailable (candidate query)')
        if event_name == 'GW170817':
            base = query_open_catalog_candidates(
                payload['window_start_mjd'],
                payload['window_end_mjd'],
                force_include_at2017gfo=True,
            )
        else:
            base = pd.DataFrame(columns=['object_id', 'survey', 'ra', 'dec', 'first_mjd', 'last_mjd', 'n_detections'])
    else:
        if cfg.catalog_backend == 'alerce':
            base = query_alerce_candidates(
                payload['window_start_mjd'],
                payload['window_end_mjd'],
                max_pages=ALERCE_MAX_PAGES,
                page_size=ALERCE_PAGE_SIZE,
                timeout_s=ALERCE_TIMEOUT_S,
                show_progress=ENABLE_CANDIDATE_PROGRESS,
            )
        elif cfg.catalog_backend == 'open_catalog':
            base = query_open_catalog_candidates(
                payload['window_start_mjd'],
                payload['window_end_mjd'],
                force_include_at2017gfo=True,
            )
        else:
            raise ValueError(f'Unsupported catalog backend: {cfg.catalog_backend}')

    if len(base) == 0:
        out = pd.DataFrame(columns=CANDIDATE_COLUMNS)
        out.to_csv(candidate_csv, index=False)
        return out

    base['first_mjd'] = pd.to_numeric(base.get('first_mjd'), errors='coerce')
    base['last_mjd'] = pd.to_numeric(base.get('last_mjd'), errors='coerce')

    start_mjd = float(payload['window_start_mjd'])
    end_mjd = float(payload['window_end_mjd'])
    has_first = base['first_mjd'].notna()
    has_last = base['last_mjd'].notna()
    base['time_window_overlap'] = (
        (has_first & has_last & (base['last_mjd'] >= start_mjd) & (base['first_mjd'] <= end_mjd))
        | (has_first & (~has_last) & base['first_mjd'].between(start_mjd, end_mjd))
        | ((~has_first) & has_last & base['last_mjd'].between(start_mjd, end_mjd))
    )

    if STRICT_WINDOW_TIME_FILTER:
        before_window = len(base)
        base = base[base['time_window_overlap']].copy()
        if ENABLE_CANDIDATE_PROGRESS:
            print(
                f"{event_name}: time-window overlap filter [{start_mjd:.3f}, {end_mjd:.3f}], "
                f"kept={len(base)}/{before_window}"
            )

    if len(base) == 0:
        out = pd.DataFrame(columns=CANDIDATE_COLUMNS)
        out.to_csv(candidate_csv, index=False)
        return out

    if 'n_detections' not in base.columns:
        base['n_detections'] = np.nan
    base['n_detections'] = pd.to_numeric(base['n_detections'], errors='coerce')

    missing_det = base['n_detections'].isna()
    if missing_det.any():
        inferred_det = np.where(
            base['first_mjd'].notna() & base['last_mjd'].notna(),
            np.where((base['last_mjd'] - base['first_mjd']).abs() > 1e-6, 2.0, 1.0),
            np.nan,
        )
        base['n_detections'] = base['n_detections'].where(~missing_det, inferred_det)

    if MIN_DETECTIONS is not None:
        before_det = len(base)
        base = base[base['n_detections'] >= float(MIN_DETECTIONS)].copy()
        if ENABLE_CANDIDATE_PROGRESS:
            print(
                f"{event_name}: min-detection(meta) filter >= {MIN_DETECTIONS}, "
                f"kept={len(base)}/{before_det}"
            )

    if len(base) == 0:
        out = pd.DataFrame(columns=CANDIDATE_COLUMNS)
        out.to_csv(candidate_csv, index=False)
        return out

    if MAX_CANDIDATES_PER_EVENT is not None and len(base) > MAX_CANDIDATES_PER_EVENT:
        base = base.iloc[:MAX_CANDIDATES_PER_EVENT].copy()

    base['event_name'] = event_name

    cred = compute_credible_levels(
        payload['gw_moc'],
        base['ra'].to_numpy(dtype=float),
        base['dec'].to_numpy(dtype=float),
        chunk=CREDIBLE_CHUNK_SIZE,
        show_progress=ENABLE_CANDIDATE_PROGRESS,
    )
    base['credible_level'] = cred
    base['in_90_region'] = base['credible_level'] <= CREDIBLE_LEVEL_MAX

    out = base[CANDIDATE_COLUMNS].copy()
    out.to_csv(candidate_csv, index=False)
    print(
        f"{event_name}: candidates={len(out)}, "
        f"in_90_region={int(out['in_90_region'].sum())}, "
        f"window_overlap={int(out['time_window_overlap'].sum())}, "
        f"median_n_det={float(out['n_detections'].median()) if len(out) else float('nan'):.1f}, "
        f"saved={candidate_csv}"
    )
    return out


candidate_tables: dict[str, pd.DataFrame] = {}
for event_name in event_payloads:
    candidate_tables[event_name] = build_candidates_for_event(event_name)

for event_name, df in candidate_tables.items():
    n_all = len(df)
    n_90 = int(df['in_90_region'].sum()) if n_all > 0 else 0
    print(f'{event_name}: total={n_all}, kept_90={n_90}')


## 4. Light Curve Download And Merge


In [ ]:
PHOTOMETRY_COLUMNS = [
    'event_id',
    'object_id',
    'survey',
    'mjd',
    'mag',
    'mag_err',
    'filter',
    'ra',
    'dec',
    'is_mock',
]


WINDOW_FILTER_COLUMNS = [
    'event_name',
    'object_id',
    'survey',
    'ra',
    'dec',
    'credible_level',
    'in_90_region',
    'window_n_detections',
]


def _row_first_float(row: dict[str, Any], keys: list[str]) -> float | None:
    for key in keys:
        if key not in row:
            continue
        value = row.get(key)
        if value in (None, '', 'nan', 'NaN', 'null', 'None'):
            continue
        try:
            return float(value)
        except Exception:
            continue
    return None


def _norm_ztf_filter(raw_filter: Any) -> str:
    if raw_filter is None:
        return 'unknown'
    raw = str(raw_filter).strip().lower()
    if raw in ('1', 'zg', 'ztf_g', 'g'):
        return 'g'
    if raw in ('2', 'zr', 'ztf_r', 'r'):
        return 'r'
    if raw in ('3', 'zi', 'ztf_i', 'i'):
        return 'i'
    return raw


def _extract_rows_from_payload(payload: Any) -> list[dict[str, Any]]:
    if payload is None:
        return []
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if isinstance(payload, dict):
        for key in ('detections', 'data', 'results', 'items', 'lightcurve', 'lc', 'result'):
            if key in payload:
                return _extract_rows_from_payload(payload[key])
        if payload and all(isinstance(v, (list, tuple)) for v in payload.values()):
            max_len = max(len(v) for v in payload.values())
            rows = []
            for idx in range(max_len):
                rows.append({k: (v[idx] if idx < len(v) else None) for k, v in payload.items()})
            return rows
    return []


def _records_to_photometry_df(records: list[dict[str, Any]], oid: str) -> pd.DataFrame:
    rows = []
    for rec in records:
        rows.append(
            {
                'event_id': None,
                'object_id': oid,
                'survey': 'ZTF',
                'mjd': float(rec['mjd']),
                'mag': float(rec['mag']),
                'mag_err': float(rec['mag_err']) if rec.get('mag_err') is not None else np.nan,
                'filter': str(rec.get('filter', '')),
                'ra': float(rec['ra']) if rec.get('ra') is not None else np.nan,
                'dec': float(rec['dec']) if rec.get('dec') is not None else np.nan,
                'is_mock': False,
            }
        )
    if not rows:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS)
    out = pd.DataFrame(rows, columns=PHOTOMETRY_COLUMNS)
    out = out.sort_values('mjd').drop_duplicates(subset=['mjd', 'filter', 'mag'], keep='first').reset_index(drop=True)
    return out


def fetch_ztf_lightcurve_from_alerce_http(oid: str, start_mjd: float, end_mjd: float) -> pd.DataFrame:
    urls = [
        f'https://api.alerce.online/ztf/v1/objects/{oid}/lightcurve',
        f'https://api.alerce.online/ztf/v1/objects/{oid}/detections',
    ]

    parsed_rows: list[dict[str, Any]] = []
    for url in urls:
        try:
            payload = http_json(url, timeout_s=ZTF_LC_TIMEOUT_S)
        except Exception:
            continue

        raw_rows = _extract_rows_from_payload(payload)
        if not raw_rows:
            continue

        for raw in raw_rows:
            low = {str(k).lower(): v for k, v in raw.items()}
            mjd = _row_first_float(low, ['mjd', 'jd', 'hjd'])
            mag = _row_first_float(low, ['magpsf', 'mag', 'magnitude', 'mag_corr', 'magap', 'psfmag'])
            if mjd is None or mag is None:
                continue
            if mjd > 2400000.5:
                mjd = mjd - 2400000.5
            if not (start_mjd <= mjd <= end_mjd):
                continue

            parsed_rows.append(
                {
                    'event_id': None,
                    'object_id': oid,
                    'survey': 'ZTF',
                    'mjd': float(mjd),
                    'mag': float(mag),
                    'mag_err': _row_first_float(low, ['sigmapsf', 'magerr', 'mag_err', 'e_magnitude', 'psfmagerr']),
                    'filter': _norm_ztf_filter(low.get('fid', low.get('filter', low.get('band')))),
                    'ra': _row_first_float(low, ['ra', 'meanra', 'objra']),
                    'dec': _row_first_float(low, ['dec', 'meandec', 'objdec']),
                    'is_mock': False,
                }
            )

        if parsed_rows:
            break

    if not parsed_rows:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS)

    out = pd.DataFrame(parsed_rows, columns=PHOTOMETRY_COLUMNS)
    out = out.sort_values('mjd').drop_duplicates(subset=['mjd', 'filter', 'mag'], keep='first').reset_index(drop=True)
    return out


def fetch_ztf_lightcurve_records(oid: str, start_mjd: float, end_mjd: float) -> pd.DataFrame:
    # Prefer public ALeRCE backend to avoid ztfquery auth failures.
    backends = []
    for backend in (ZTF_LC_BACKEND, 'alerce', 'auto'):
        b = str(backend).strip().lower()
        if b and b not in backends:
            backends.append(b)

    for backend in backends:
        try:
            records = ZTF_SCRIPT.fetch_lightcurve(
                oid=oid,
                backend=backend,
                timeout_s=ZTF_LC_TIMEOUT_S,
                min_mjd=start_mjd,
                max_mjd=end_mjd,
                bands=None,
            )
            if records:
                return _records_to_photometry_df(records, oid)
        except Exception:
            continue

    # Last resort: raw HTTP against ALeRCE endpoints.
    return fetch_ztf_lightcurve_from_alerce_http(oid, start_mjd, end_mjd)


def fetch_open_catalog_photometry(object_id: str, start_mjd: float, end_mjd: float) -> pd.DataFrame:
    urls = [
        f'https://api.astrocats.space/{object_id}/photometry/time+magnitude+e_magnitude+band+telescope?format=json',
        f'https://api.astrocats.space/{object_id}?format=json',
    ]

    payload = None
    for url in urls:
        try:
            payload = http_json(url)
            break
        except Exception:
            continue

    if payload is None:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS)

    records = []

    def walk(obj: Any):
        if isinstance(obj, dict):
            if 'time' in obj and ('magnitude' in obj or 'mag' in obj):
                records.append(obj)
            for value in obj.values():
                walk(value)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(payload)

    rows = []
    for row in records:
        t_val = row.get('time')
        m_val = row.get('magnitude', row.get('mag'))
        if t_val is None or m_val is None:
            continue
        try:
            time_val = float(t_val)
            mag_val = float(m_val)
        except Exception:
            continue
        if time_val > 2400000.5:
            time_val -= 2400000.5
        if not (start_mjd <= time_val <= end_mjd):
            continue

        mag_err = row.get('e_magnitude', row.get('mag_err'))
        try:
            mag_err = float(mag_err) if mag_err is not None else np.nan
        except Exception:
            mag_err = np.nan

        rows.append(
            {
                'event_id': None,
                'object_id': object_id,
                'survey': str(row.get('telescope', row.get('instrument', 'OpenCatalog'))),
                'mjd': time_val,
                'mag': mag_val,
                'mag_err': mag_err,
                'filter': str(row.get('band', row.get('filter', 'unknown'))),
                'ra': np.nan,
                'dec': np.nan,
                'is_mock': False,
            }
        )

    return pd.DataFrame(rows, columns=PHOTOMETRY_COLUMNS)


def build_offline_mock_photometry(event_name: str, object_id: str, t0_mjd: float, ra: float, dec: float) -> pd.DataFrame:
    # Fallback only when online APIs are unavailable and no cache exists.
    # This keeps the pipeline executable, but should not be interpreted as real observations.
    mjd = np.array([t0_mjd - 0.4, t0_mjd + 0.6, t0_mjd + 1.2, t0_mjd + 2.1, t0_mjd + 4.5])
    mag = np.array([17.40, 17.10, 17.45, 17.90, 18.80])
    err = np.array([0.07, 0.06, 0.08, 0.10, 0.14])
    flt = ['g', 'r', 'i', 'r', 'z']

    rows = []
    for m, y, e, f in zip(mjd, mag, err, flt):
        rows.append(
            {
                'event_id': event_name,
                'object_id': object_id,
                'survey': 'offline_mock',
                'mjd': float(m),
                'mag': float(y),
                'mag_err': float(e),
                'filter': f,
                'ra': float(ra),
                'dec': float(dec),
                'is_mock': True,
            }
        )
    return pd.DataFrame(rows, columns=PHOTOMETRY_COLUMNS)


def _apply_window_photometry_constraints(
    obj_df: pd.DataFrame,
    start_mjd: float,
    end_mjd: float,
    min_detections: int | None = MIN_DETECTIONS,
) -> tuple[pd.DataFrame, int]:
    if len(obj_df) == 0:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), 0

    g = obj_df.copy()
    g['mjd'] = pd.to_numeric(g.get('mjd'), errors='coerce')
    g['mag'] = pd.to_numeric(g.get('mag'), errors='coerce')
    g['mag_err'] = pd.to_numeric(g.get('mag_err'), errors='coerce')

    g = g[np.isfinite(g['mjd']) & np.isfinite(g['mag'])]
    g = g[(g['mjd'] >= start_mjd) & (g['mjd'] <= end_mjd)].copy()
    if len(g) == 0:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), 0

    g = g.sort_values('mjd').drop_duplicates(subset=['mjd', 'filter', 'mag'], keep='first').reset_index(drop=True)
    n_window = int(len(g))

    if STRICT_WINDOW_MIN_DETECTIONS and min_detections is not None and n_window < int(min_detections):
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), n_window

    return g[PHOTOMETRY_COLUMNS].copy(), n_window


def _filter_cached_merged(
    merged: pd.DataFrame,
    start_mjd: float,
    end_mjd: float,
) -> tuple[pd.DataFrame, dict[str, int]]:
    if len(merged) == 0:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), {}

    if REAL_DATA_ONLY and 'is_mock' in merged.columns:
        merged = merged[merged['is_mock'] != True].copy()
        if len(merged) == 0:
            return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), {}

    kept: list[pd.DataFrame] = []
    n_det_map: dict[str, int] = {}
    for oid, grp in merged.groupby('object_id'):
        fgrp, n_window = _apply_window_photometry_constraints(grp, start_mjd, end_mjd)
        if len(fgrp) == 0:
            continue
        oid_s = str(oid)
        kept.append(fgrp)
        n_det_map[oid_s] = int(n_window)

    if not kept:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS), {}

    out = pd.concat(kept, ignore_index=True)
    out = out.sort_values(['object_id', 'mjd']).reset_index(drop=True)
    return out[PHOTOMETRY_COLUMNS].copy(), n_det_map


def _write_window_filtered_candidates(
    event_name: str,
    event_dir: Path,
    candidates: pd.DataFrame,
    n_det_map: dict[str, int],
) -> pd.DataFrame:
    if len(candidates) == 0 or not n_det_map:
        selected = pd.DataFrame(columns=WINDOW_FILTER_COLUMNS)
    else:
        temp = candidates.copy()
        temp['object_id'] = temp['object_id'].astype(str)
        temp = temp[temp['object_id'].isin(set(n_det_map.keys()))].copy()
        if len(temp) == 0:
            selected = pd.DataFrame(columns=WINDOW_FILTER_COLUMNS)
        else:
            temp['window_n_detections'] = temp['object_id'].map(n_det_map).astype(int)
            cols = [
                'event_name',
                'object_id',
                'survey',
                'ra',
                'dec',
                'credible_level',
                'in_90_region',
                'window_n_detections',
            ]
            selected = temp[cols].drop_duplicates(subset=['object_id']).reset_index(drop=True)

    out_csv = event_dir / f'candidates_{event_name}_window_filtered.csv'
    selected.to_csv(out_csv, index=False)
    print(f"{event_name}: saved window-filtered candidates ({len(selected)}) -> {out_csv}")
    return selected


def load_or_build_merged_photometry(event_name: str, force_refresh: bool = FORCE_REFRESH) -> pd.DataFrame:
    payload = event_payloads[event_name]
    candidates = candidate_tables.get(event_name)
    if candidates is None:
        return pd.DataFrame(columns=PHOTOMETRY_COLUMNS)

    event_dir = CACHE_ROOT / event_name
    raw_dir = event_dir / 'raw_lc'
    raw_dir.mkdir(parents=True, exist_ok=True)
    merged_parquet = event_dir / f'merged_lc_{event_name}.parquet'
    merged_csv = event_dir / f'merged_lc_{event_name}.csv'

    start_mjd = float(payload['window_start_mjd'])
    end_mjd = float(payload['window_end_mjd'])

    if not force_refresh:
        cached = None
        if merged_parquet.exists():
            try:
                cached = pd.read_parquet(merged_parquet)
                print(f'{event_name}: loaded merged photometry cache ({merged_parquet})')
            except Exception:
                cached = None
        if cached is None and merged_csv.exists():
            cached = pd.read_csv(merged_csv)
            print(f'{event_name}: loaded merged photometry cache ({merged_csv})')

        if cached is not None:
            cached_filtered, n_det_map = _filter_cached_merged(cached, start_mjd, end_mjd)
            _write_window_filtered_candidates(event_name, event_dir, candidates, n_det_map)
            return cached_filtered

    use_candidates = candidates[candidates['in_90_region'] == True].copy()
    if STRICT_WINDOW_TIME_FILTER and 'time_window_overlap' in use_candidates.columns:
        use_candidates = use_candidates[use_candidates['time_window_overlap'] == True].copy()

    if len(use_candidates) == 0:
        print(f'{event_name}: no candidates inside 90% region after time-window filtering')
        empty = pd.DataFrame(columns=PHOTOMETRY_COLUMNS)
        empty.to_csv(merged_csv, index=False)
        _write_window_filtered_candidates(event_name, event_dir, candidates, {})
        return empty

    if MAX_CANDIDATES_PER_EVENT is not None and len(use_candidates) > MAX_CANDIDATES_PER_EVENT:
        use_candidates = use_candidates.iloc[:MAX_CANDIDATES_PER_EVENT].copy()

    all_rows: list[pd.DataFrame] = []
    n_det_map: dict[str, int] = {}
    net_ok = network_available()

    n_download_fail = 0
    n_low_det = 0

    for _, cand in use_candidates.iterrows():
        object_id = str(cand['object_id'])
        survey = str(cand.get('survey', ''))

        object_raw_csv = raw_dir / f"{object_id.replace('/', '_')}.csv"

        obj_df = pd.DataFrame(columns=PHOTOMETRY_COLUMNS)
        if object_raw_csv.exists() and not force_refresh:
            try:
                obj_df = pd.read_csv(object_raw_csv)
            except Exception:
                obj_df = pd.DataFrame(columns=PHOTOMETRY_COLUMNS)

        if len(obj_df) == 0:
            if net_ok:
                try:
                    if object_id.startswith('ZTF') or survey.upper() == 'ZTF':
                        obj_df = fetch_ztf_lightcurve_records(object_id, start_mjd, end_mjd)
                    else:
                        obj_df = fetch_open_catalog_photometry(object_id, start_mjd, end_mjd)
                except Exception as exc:
                    n_download_fail += 1
                    if ENABLE_CANDIDATE_PROGRESS:
                        print(f'{event_name} {object_id}: download failed ({exc})')
            else:
                if ENABLE_CANDIDATE_PROGRESS:
                    print(f'{event_name} {object_id}: SKIPPED: network unavailable (photometry)')

        if len(obj_df) == 0 and ENABLE_OFFLINE_MOCK_IF_EMPTY and event_name == 'GW170817' and object_id == 'AT2017gfo':
            obj_df = build_offline_mock_photometry(
                event_name=event_name,
                object_id=object_id,
                t0_mjd=payload['config'].t0_mjd,
                ra=float(cand['ra']),
                dec=float(cand['dec']),
            )

        if REAL_DATA_ONLY and 'is_mock' in obj_df.columns:
            obj_df = obj_df[obj_df['is_mock'] != True].copy()

        filtered_obj_df, n_window = _apply_window_photometry_constraints(obj_df, start_mjd, end_mjd)
        if len(filtered_obj_df) == 0:
            if n_window > 0:
                n_low_det += 1
            continue

        filtered_obj_df['event_id'] = event_name
        filtered_obj_df['object_id'] = object_id
        if 'ra' in filtered_obj_df and filtered_obj_df['ra'].isna().all():
            filtered_obj_df['ra'] = float(cand['ra'])
        if 'dec' in filtered_obj_df and filtered_obj_df['dec'].isna().all():
            filtered_obj_df['dec'] = float(cand['dec'])

        filtered_obj_df = filtered_obj_df[PHOTOMETRY_COLUMNS].copy()
        filtered_obj_df.to_csv(object_raw_csv, index=False)

        all_rows.append(filtered_obj_df)
        n_det_map[object_id] = int(n_window)

    if len(all_rows) == 0:
        merged = pd.DataFrame(columns=PHOTOMETRY_COLUMNS)
    else:
        merged = pd.concat(all_rows, ignore_index=True)
        merged = merged.sort_values(['object_id', 'mjd']).reset_index(drop=True)

    selected_df = _write_window_filtered_candidates(event_name, event_dir, candidates, n_det_map)
    keep_ids = set(selected_df['object_id'].astype(str).tolist())
    if keep_ids:
        candidate_tables[event_name] = candidates[candidates['object_id'].astype(str).isin(keep_ids)].copy()

    try:
        merged.to_parquet(merged_parquet, index=False)
        print(f'{event_name}: saved merged photometry to {merged_parquet}')
    except Exception as exc:
        print(f'{event_name}: parquet save skipped ({exc})')
    merged.to_csv(merged_csv, index=False)

    if ENABLE_CANDIDATE_PROGRESS:
        print(
            f"{event_name}: photometry summary kept={len(n_det_map)}, "
            f"download_fail={n_download_fail}, low_det_drop={n_low_det}"
        )

    return merged


merged_photometry: dict[str, pd.DataFrame] = {}
for event_name in event_payloads:
    merged_photometry[event_name] = load_or_build_merged_photometry(event_name)
    print(f"{event_name}: merged rows={len(merged_photometry[event_name])}")


## 5. Model Input Transform (Sparse Sequence, Length 90)


In [ ]:
from typing import NamedTuple


class InputPack(NamedTuple):
    gw_scalar: torch.Tensor      # [N, 7]
    gw_moc: torch.Tensor         # [N, 7, 19200]
    opt_values: torch.Tensor     # [N, 90, 6]
    opt_errors: torch.Tensor     # [N, 90, 6]
    opt_masks: torch.Tensor      # [N, 90, 6]
    opt_times: torch.Tensor      # [N, 90]
    opt_coords: torch.Tensor     # [N, 2]
    meta: pd.DataFrame


FILTER_ALIASES = {
    'LSST-u': {'u', 'lsst-u', 'ztf_u', 'sdssu', 'up', 'u_ps1'},
    'LSST-g': {'g', 'zg', 'lsst-g', 'ztf_g', 'sdssg', 'gp', 'g_ps1', 'f475w', 'atlas_c', 'c'},
    'LSST-r': {'r', 'zr', 'lsst-r', 'ztf_r', 'sdssr', 'rp', 'r_ps1', 'f625w', 'atlas_o', 'o'},
    'LSST-i': {'i', 'zi', 'lsst-i', 'ztf_i', 'sdssi', 'ip', 'i_ps1', 'f775w'},
    'LSST-z': {'z', 'lsst-z', 'sdssz', 'zp', 'z_ps1', 'f850lp'},
    'LSST-Y': {'y', 'lsst-y', 'yp', 'y_ps1', 'f105w'},
}


def normalize_to_lsst_band(filter_value: Any) -> str | None:
    if filter_value is None:
        return None
    raw = str(filter_value).strip().lower()
    for lsst_band, aliases in FILTER_ALIASES.items():
        if raw in aliases:
            return lsst_band
    return None


def mag_to_fluxcal(mag: np.ndarray, mag_err: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # SNANA-compatible flux scale, zeropoint=27.5
    flux = 10.0 ** (-0.4 * (mag - 27.5))
    flux_err = np.abs(flux * np.log(10.0) * 0.4 * mag_err)
    return flux, flux_err


def build_input_pack_for_event(event_name: str, seq_len: int = SEQ_LEN) -> InputPack | None:
    payload = event_payloads[event_name]
    cfg: EventConfig = payload['config']
    merged = merged_photometry.get(event_name)
    candidates = candidate_tables.get(event_name)
    if merged is None or len(merged) == 0:
        print(f'{event_name}: no merged photometry available for tensor conversion')
        return None

    if candidates is not None and len(candidates) > 0:
        cand_coords = (
            candidates[['object_id', 'ra', 'dec', 'credible_level', 'in_90_region']]
            .drop_duplicates(subset=['object_id'])
            .set_index('object_id')
        )
    else:
        cand_coords = pd.DataFrame(columns=['ra', 'dec', 'credible_level', 'in_90_region'])

    values_list: list[np.ndarray] = []
    errors_list: list[np.ndarray] = []
    masks_list: list[np.ndarray] = []
    times_list: list[np.ndarray] = []
    coords_list: list[np.ndarray] = []
    meta_rows: list[dict[str, Any]] = []

    start_mjd = cfg.t0_mjd - WINDOW_PRE_DAYS
    end_mjd = cfg.t0_mjd + WINDOW_POST_DAYS

    for object_id, grp in merged.groupby('object_id'):
        g = grp.copy()
        g = g[(g['mjd'] >= start_mjd) & (g['mjd'] <= end_mjd)].copy()
        if len(g) == 0:
            continue

        if REAL_DATA_ONLY and 'is_mock' in g.columns:
            g = g[g['is_mock'] != True].copy()
            if len(g) == 0:
                continue

        g['lsst_band'] = g['filter'].map(normalize_to_lsst_band)
        g = g[g['lsst_band'].notna()].copy()
        if len(g) == 0:
            continue

        g = g.sort_values('mjd').reset_index(drop=True)

        mag = g['mag'].to_numpy(dtype=float)
        mag_err = g['mag_err'].to_numpy(dtype=float)
        mag_err = np.where(np.isfinite(mag_err) & (mag_err > 0), mag_err, 0.15)

        flux, flux_err = mag_to_fluxcal(mag, mag_err)
        if not np.isfinite(flux).any():
            continue

        mean_flux = float(np.mean(flux))
        std_flux = float(np.std(flux))
        if not np.isfinite(std_flux) or std_flux < 1e-8:
            std_flux = 1.0

        flux_n = (flux - mean_flux) / std_flux
        flux_err_n = flux_err / std_flux
        rel_time = (g['mjd'].to_numpy(dtype=float) - cfg.t0_mjd) / 100.0

        val_mat = np.zeros((seq_len, len(LSST_BANDS)), dtype=np.float32)
        err_mat = np.zeros((seq_len, len(LSST_BANDS)), dtype=np.float32)
        mask_mat = np.zeros((seq_len, len(LSST_BANDS)), dtype=np.float32)
        time_vec = np.zeros((seq_len,), dtype=np.float32)

        used = 0
        for obs_idx in range(min(len(g), seq_len)):
            lsst_band = g.loc[obs_idx, 'lsst_band']
            band_idx = BAND_TO_INDEX[lsst_band]
            val_mat[obs_idx, band_idx] = flux_n[obs_idx]
            err_mat[obs_idx, band_idx] = flux_err_n[obs_idx]
            mask_mat[obs_idx, band_idx] = 1.0
            time_vec[obs_idx] = rel_time[obs_idx]
            used += 1

        if used == 0:
            continue

        if np.isfinite(g['ra']).any() and np.isfinite(g['dec']).any():
            ra = float(np.nanmedian(g['ra']))
            dec = float(np.nanmedian(g['dec']))
        elif object_id in cand_coords.index:
            ra = float(cand_coords.loc[object_id, 'ra'])
            dec = float(cand_coords.loc[object_id, 'dec'])
        else:
            continue

        values_list.append(val_mat)
        errors_list.append(err_mat)
        masks_list.append(mask_mat)
        times_list.append(time_vec)
        coords_list.append(np.array([ra, dec], dtype=np.float32))

        cred = float(cand_coords.loc[object_id, 'credible_level']) if object_id in cand_coords.index else np.nan
        in90 = bool(cand_coords.loc[object_id, 'in_90_region']) if object_id in cand_coords.index else False
        meta_rows.append(
            {
                'event_name': event_name,
                'object_id': object_id,
                'survey': str(g['survey'].iloc[0]),
                'n_points_original': int(len(g)),
                'n_points_used': int(min(len(g), seq_len)),
                'credible_level': cred,
                'in_90_region': in90,
                'is_mock': bool(g['is_mock'].any()),
            }
        )

    if len(values_list) == 0:
        return None

    values = torch.tensor(np.stack(values_list, axis=0), dtype=torch.float32)
    errors = torch.tensor(np.stack(errors_list, axis=0), dtype=torch.float32)
    masks = torch.tensor(np.stack(masks_list, axis=0), dtype=torch.float32)
    times = torch.tensor(np.stack(times_list, axis=0), dtype=torch.float32)
    coords = torch.tensor(np.stack(coords_list, axis=0), dtype=torch.float32)

    n = values.shape[0]
    gw_scalar = payload['gw_scalar'].unsqueeze(0).repeat(n, 1)
    gw_moc = payload['gw_moc'].unsqueeze(0).repeat(n, 1, 1)

    assert values.shape == (n, seq_len, len(LSST_BANDS))
    assert errors.shape == values.shape
    assert masks.shape == values.shape
    assert times.shape == (n, seq_len)
    assert coords.shape == (n, 2)

    observed_time = times[masks.sum(dim=-1) > 0]
    if observed_time.numel() > 0:
        assert float(observed_time.min()) >= -0.3001
        assert float(observed_time.max()) <= 0.6001

    meta_df = pd.DataFrame(meta_rows)
    return InputPack(
        gw_scalar=gw_scalar,
        gw_moc=gw_moc,
        opt_values=values,
        opt_errors=errors,
        opt_masks=masks,
        opt_times=times,
        opt_coords=coords,
        meta=meta_df,
    )


# Offline structural tests (optional)
if RUN_OFFLINE_STRUCT_TESTS:
    mock_rows = []
    t0_test = EVENT_CONFIGS['GW170817'].t0_mjd
    for i in range(12):
        mock_rows.append(
            {
                'event_id': 'GW170817',
                'object_id': 'MOCK_SHORT',
                'survey': 'test',
                'mjd': t0_test - 2.0 + i * 0.3,
                'mag': 18.0 + 0.02 * i,
                'mag_err': 0.08,
                'filter': ['g', 'r', 'i'][i % 3],
                'ra': 197.45,
                'dec': -23.38,
                'is_mock': True,
            }
        )
    for i in range(260):
        mock_rows.append(
            {
                'event_id': 'GW170817',
                'object_id': 'MOCK_LONG',
                'survey': 'test',
                'mjd': t0_test - 9.0 + i * 0.1,
                'mag': 19.0 + 0.01 * np.sin(i / 6.0),
                'mag_err': 0.10,
                'filter': ['g', 'r', 'i', 'z'][i % 4],
                'ra': 197.55,
                'dec': -23.30,
                'is_mock': True,
            }
        )

    mock_df = pd.DataFrame(mock_rows)
    real_backup = merged_photometry.get('GW170817')
    cand_backup = candidate_tables.get('GW170817')
    try:
        merged_photometry['GW170817'] = mock_df
        candidate_tables['GW170817'] = pd.DataFrame(
            [
                {
                    'event_name': 'GW170817',
                    'object_id': 'MOCK_SHORT',
                    'survey': 'test',
                    'ra': 197.45,
                    'dec': -23.38,
                    'first_mjd': t0_test - 2,
                    'last_mjd': t0_test + 2,
                    'time_window_overlap': True,
                    'n_detections': 12,
                    'credible_level': 0.5,
                    'in_90_region': True,
                },
                {
                    'event_name': 'GW170817',
                    'object_id': 'MOCK_LONG',
                    'survey': 'test',
                    'ra': 197.55,
                    'dec': -23.30,
                    'first_mjd': t0_test - 5,
                    'last_mjd': t0_test + 20,
                    'time_window_overlap': True,
                    'n_detections': 200,
                    'credible_level': 0.6,
                    'in_90_region': True,
                },
            ]
        )

        mock_pack = build_input_pack_for_event('GW170817', seq_len=SEQ_LEN)
        assert mock_pack is not None
        assert mock_pack.opt_values.shape[0] == 2
        assert mock_pack.opt_values.shape[1:] == (90, 6)

        meta_by_oid = mock_pack.meta.set_index('object_id')
        long_orig = int(meta_by_oid.loc['MOCK_LONG', 'n_points_original'])
        long_used = int(meta_by_oid.loc['MOCK_LONG', 'n_points_used'])
        short_used = int(meta_by_oid.loc['MOCK_SHORT', 'n_points_used'])

        assert long_orig > SEQ_LEN
        assert long_used == SEQ_LEN
        assert short_used < SEQ_LEN
        print('Offline structural tests passed.')
    finally:
        if real_backup is not None:
            merged_photometry['GW170817'] = real_backup
        else:
            merged_photometry.pop('GW170817', None)
        if cand_backup is not None:
            candidate_tables['GW170817'] = cand_backup
        else:
            candidate_tables.pop('GW170817', None)
else:
    print('Offline structural tests skipped (RUN_OFFLINE_STRUCT_TESTS=False).')

input_packs: dict[str, InputPack] = {}
for event_name in event_payloads:
    pack = build_input_pack_for_event(event_name, seq_len=SEQ_LEN)
    if pack is None:
        print(f'{event_name}: no valid tensors built')
        continue
    input_packs[event_name] = pack
    print(f"{event_name}: tensors built for N={pack.opt_values.shape[0]} candidates")


## 6. Model Inference


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIG_PATH = MODEL_DIR / 'args' / 'ALBEF_supcon_v2.json'
CKPT_PATH = PROJECT_ROOT / 'data' / 'model' / 'checkpoints' / 'supcon_v2' / 'ALBEF' / 'albef_best.pth'


def load_trained_model(config_path: Path, ckpt_path: Path, device: torch.device):
    if not config_path.exists():
        raise FileNotFoundError(config_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)

    with config_path.open('r', encoding='utf-8') as f:
        cfg = json.load(f)

    model = GWOpticalALBEFModel(
        gw_scalar_dim=7,
        gw_skymap_channels=7,
        optical_input_dim=6,
        ref_time_dim=cfg.get('ref_dim', 64),
        enc_dim=cfg.get('enc_dim', 128),
        proj_dim=cfg.get('proj_dim', 256),
        fusion_attn_dim=cfg.get('fusion_attn_dim'),
        fusion_hidden_dim=cfg.get('fusion_hidden_dim'),
        temp_init=cfg.get('temp_init', 0.07),
        temp_min=cfg.get('temp_min', 0.01),
        temp_max=cfg.get('temp_max', 100.0),
        fusion_dropout=cfg.get('fusion_dropout', 0.1),
        gw_dropout=cfg.get('gw_dropout', 0.1),
        opt_dropout=cfg.get('opt_dropout', 0.1),
        proj_dropout=cfg.get('proj_dropout', 0.0),
        feature_dropout=cfg.get('feature_dropout', 0.0),
        label_smoothing=cfg.get('label_smoothing', 0.0),
        itc_label_smoothing=cfg.get('itc_label_smoothing', 0.0),
        use_lightweight_gw=cfg.get('use_lightweight_gw', False),
        dual_fusion=cfg.get('dual_fusion', True),
    ).to(device)

    try:
        checkpoint = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(str(ckpt_path), map_location=device)
    state = checkpoint.get('model_state_dict', checkpoint)
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f'Missing keys: {missing[:5]} ... total={len(missing)}')
    if unexpected:
        print(f'Unexpected keys: {unexpected[:5]} ... total={len(unexpected)}')

    model.eval()
    return model, cfg


def build_ref_time(batch_size: int, cfg: dict[str, Any], device: torch.device) -> torch.Tensor:
    n_ref = int(cfg.get('n_ref', 64))
    ref_start = float(cfg.get('ref_start', -0.3))
    ref_end = float(cfg.get('ref_end', 0.6))
    ref = torch.linspace(ref_start, ref_end, n_ref, dtype=torch.float32, device=device)
    return ref.unsqueeze(0).repeat(batch_size, 1)


def run_inference(event_name: str, pack: InputPack, model, cfg: dict[str, Any], batch_size: int = 256) -> pd.DataFrame:
    n_total = pack.opt_values.shape[0]
    pred_rows = []

    with torch.no_grad():
        for start in range(0, n_total, batch_size):
            end = min(start + batch_size, n_total)
            b = end - start

            gw_s = pack.gw_scalar[start:end].to(DEVICE)
            gw_m = pack.gw_moc[start:end].to(DEVICE)
            ov = pack.opt_values[start:end].to(DEVICE)
            oe = pack.opt_errors[start:end].to(DEVICE)
            om = pack.opt_masks[start:end].to(DEVICE)
            ot = pack.opt_times[start:end].to(DEVICE)
            oc = pack.opt_coords[start:end].to(DEVICE)
            opt_ref_t = build_ref_time(b, cfg, DEVICE)

            g, z_l, h_l, H_gw = model.encode(gw_s, gw_m, oc, ot, ov, opt_ref_t, om, oe)
            cred = compute_credible_level(gw_m, oc)
            logits = model.fusion_logits(g, h_l, z_l=z_l, H_gw=H_gw, cred_level=cred)

            assert torch.isfinite(logits).all(), f'{event_name}: non-finite logits detected'

            probs = torch.softmax(logits, dim=1)[:, 1]
            feat_g = F.normalize(model.gw_proj(g), p=2, dim=1, eps=1e-8)
            feat_o = F.normalize(model.opt_proj(z_l), p=2, dim=1, eps=1e-8)
            cosine = (feat_g * feat_o).sum(dim=1)

            for i in range(b):
                meta_row = pack.meta.iloc[start + i]
                pred_rows.append(
                    {
                        'event_name': event_name,
                        'object_id': meta_row['object_id'],
                        'survey': meta_row['survey'],
                        'n_points_original': int(meta_row['n_points_original']),
                        'n_points_used': int(meta_row['n_points_used']),
                        'candidate_credible_level': float(meta_row['credible_level']) if pd.notna(meta_row['credible_level']) else np.nan,
                        'in_90_region': bool(meta_row['in_90_region']),
                        'is_mock': bool(meta_row['is_mock']),
                        'match_prob': float(probs[i].item()),
                        'cosine_sim': float(cosine[i].item()),
                        'model_credible_level': float(cred[i].item()),
                    }
                )

    out = pd.DataFrame(pred_rows)
    out = out.sort_values('match_prob', ascending=False).reset_index(drop=True)
    return out


model, model_cfg = load_trained_model(CONFIG_PATH, CKPT_PATH, DEVICE)
print(f'Model loaded on device: {DEVICE}')

# Smoke test: 1 real/mocked batch + finite check
smoke_event = 'GW170817' if 'GW170817' in input_packs else next(iter(input_packs.keys()), None)
if smoke_event is not None:
    smoke_pack = input_packs[smoke_event]
    smoke_n = min(2, smoke_pack.opt_values.shape[0])
    smoke_subset = InputPack(
        gw_scalar=smoke_pack.gw_scalar[:smoke_n],
        gw_moc=smoke_pack.gw_moc[:smoke_n],
        opt_values=smoke_pack.opt_values[:smoke_n],
        opt_errors=smoke_pack.opt_errors[:smoke_n],
        opt_masks=smoke_pack.opt_masks[:smoke_n],
        opt_times=smoke_pack.opt_times[:smoke_n],
        opt_coords=smoke_pack.opt_coords[:smoke_n],
        meta=smoke_pack.meta.iloc[:smoke_n].reset_index(drop=True),
    )
    smoke_pred = run_inference(smoke_event, smoke_subset, model, model_cfg, batch_size=2)
    assert len(smoke_pred) == smoke_n
    assert np.isfinite(smoke_pred['match_prob']).all()
    assert np.isfinite(smoke_pred['cosine_sim']).all()
    print('Model forward smoke test passed.')
else:
    print('Smoke test skipped: no input pack available.')


event_predictions: dict[str, pd.DataFrame] = {}
for event_name, pack in input_packs.items():
    pred_df = run_inference(event_name, pack, model, model_cfg, batch_size=256)
    if REAL_DATA_ONLY and 'is_mock' in pred_df.columns:
        pred_df = pred_df[pred_df['is_mock'] == False].reset_index(drop=True)
    event_predictions[event_name] = pred_df

    out_csv = CACHE_ROOT / event_name / f'predictions_{event_name}.csv'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_csv(out_csv, index=False)

    print(f"{event_name}: saved {len(pred_df)} predictions -> {out_csv}")


## 7. Results Summary And Visualization


In [ ]:
def plot_topk_lightcurves(event_name: str, pred_df: pd.DataFrame, phot_df: pd.DataFrame, top_k: int = 5):
    if len(pred_df) == 0 or len(phot_df) == 0:
        print(f'{event_name}: no data for top-k lightcurve plot')
        return

    top = pred_df.head(top_k)
    fig, axes = plt.subplots(top_k, 1, figsize=(10, max(3, 2.2 * top_k)), sharex=True)
    if top_k == 1:
        axes = [axes]

    t0_mjd = EVENT_CONFIGS[event_name].t0_mjd if event_name in EVENT_CONFIGS else None

    color_map = {
        'g': '#2ca02c',
        'r': '#d62728',
        'i': '#ff7f0e',
        'z': '#1f77b4',
        'y': '#9467bd',
        'u': '#17becf',
    }

    for ax, (_, row) in zip(axes, top.iterrows()):
        oid = row['object_id']
        g = phot_df[phot_df['object_id'] == oid].copy()
        if len(g) == 0:
            ax.set_title(f'{oid}: no photometry rows')
            continue
        for filt, fg in g.groupby(g['filter'].astype(str).str.lower()):
            c = color_map.get(filt[:1], '#444444')
            ax.errorbar(
                fg['mjd'],
                fg['mag'],
                yerr=fg['mag_err'],
                fmt='o',
                ms=3,
                alpha=0.85,
                color=c,
                label=filt,
            )

        if t0_mjd is not None:
            ax.axvline(t0_mjd, color='black', linestyle='--', linewidth=1.2, alpha=0.9, label='GW t0')

        ax.invert_yaxis()
        ax.grid(alpha=0.25)
        ax.set_ylabel('mag')
        ax.set_title(
            f"{oid} | p={row['match_prob']:.4f} | cos={row['cosine_sim']:.4f} | cred={row['model_credible_level']:.4f}"
        )
        ax.legend(loc='best', fontsize=8)

    axes[-1].set_xlabel('MJD')
    fig.tight_layout()
    plt.show()


def plot_event_results(event_name: str, pred_df: pd.DataFrame, phot_df: pd.DataFrame):
    if len(pred_df) == 0:
        print(f'{event_name}: no predictions to plot')
        return

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    axes[0].hist(pred_df['match_prob'], bins=25, alpha=0.85, color='#1f77b4')
    axes[0].set_title(f'{event_name}: Match Probability Distribution')
    axes[0].set_xlabel('match_prob')
    axes[0].set_ylabel('count')
    axes[0].grid(alpha=0.25)

    axes[1].scatter(
        pred_df['model_credible_level'],
        pred_df['match_prob'],
        s=18,
        alpha=0.75,
        c=pred_df['is_mock'].map({True: '#d62728', False: '#2ca02c'}),
    )
    axes[1].set_title(f'{event_name}: match_prob vs model_credible_level')
    axes[1].set_xlabel('model_credible_level')
    axes[1].set_ylabel('match_prob')
    axes[1].grid(alpha=0.25)

    fig.tight_layout()
    plt.show()

    plot_topk_lightcurves(event_name, pred_df, phot_df, top_k=min(10, len(pred_df)))


summary_rows = []
for event_name in sorted(event_predictions.keys()):
    pred_df = event_predictions[event_name]
    phot_df = merged_photometry.get(event_name, pd.DataFrame(columns=PHOTOMETRY_COLUMNS))

    if REAL_DATA_ONLY:
        if 'is_mock' in pred_df.columns:
            pred_df = pred_df[pred_df['is_mock'] == False].reset_index(drop=True)
        if 'is_mock' in phot_df.columns:
            phot_df = phot_df[phot_df['is_mock'] != True].reset_index(drop=True)

    summary_rows.append(
        {
            'event_name': event_name,
            'n_candidates_inferred': int(len(pred_df)),
            'n_mock_candidates': int(pred_df['is_mock'].sum()) if len(pred_df) else 0,
            'top_match_prob': float(pred_df['match_prob'].max()) if len(pred_df) else np.nan,
            'median_match_prob': float(pred_df['match_prob'].median()) if len(pred_df) else np.nan,
            'mean_credible_level': float(pred_df['model_credible_level'].mean()) if len(pred_df) else np.nan,
        }
    )

    print(f'===== {event_name} =====')
    display(pred_df.head(10))
    plot_event_results(event_name, pred_df, phot_df)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

all_predictions = pd.concat(event_predictions.values(), ignore_index=True) if event_predictions else pd.DataFrame()
if len(all_predictions):
    global_csv = CACHE_ROOT / 'predictions_all_events.csv'
    all_predictions.to_csv(global_csv, index=False)
    print(f'Saved combined predictions: {global_csv}')
